In [ ]:
pip install bertopic sentence-transformers

In [12]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF

# 1. Cargar el dataset inmaculado
df = pd.read_parquet("../data/intervenciones_limpias.parquet")
textos = df['texto_limpio'].dropna().tolist()

print(f"Entrenando baselines con {len(textos)} documentos...")

# 2. Vectorizar (Ya no pasamos la lista gigante acá porque el parquet está limpio)
# Mantenemos las stopwords genéricas de sklearn como red de seguridad
print("Vectorizando para LDA...")
tf_vectorizer = CountVectorizer(max_df=0.85, min_df=15)
tf_matrix = tf_vectorizer.fit_transform(textos)
tf_feature_names = tf_vectorizer.get_feature_names_out()

print("Vectorizando para NMF...")
tfidf_vectorizer = TfidfVectorizer(max_df=0.85, min_df=15)
tfidf_matrix = tfidf_vectorizer.fit_transform(textos)
tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()

# 3. Función para imprimir resultados
def print_top_words(model, feature_names, n_top_words=10):
    for topic_idx, topic in enumerate(model.components_):
        message = f"Tópico #{topic_idx}: "
        message += " ".join([feature_names[i] for i in topic.argsort()[:-n_top_words - 1:-1]])
        print(message)
    print()

k_topicos = 15

# 4. Entrenar modelos
print("Entrenando LDA...")
lda_model = LatentDirichletAllocation(n_components=k_topicos, random_state=42, n_jobs=-1)
lda_model.fit(tf_matrix)
print("\n--- Resultados Finales LDA ---")
print_top_words(lda_model, tf_feature_names)

print("Entrenando NMF...")
nmf_model = NMF(n_components=k_topicos, random_state=42, init='nndsvda')
nmf_model.fit(tfidf_matrix)
print("\n--- Resultados Finales NMF ---")
print_top_words(nmf_model, tfidf_feature_names)

Entrenando baselines con 172417 documentos...
Vectorizando para LDA...
Vectorizando para NMF...
Entrenando LDA...

--- Resultados Finales LDA ---
Tópico #0: 2006 diputados 2007 sartori aignasse vargas stella narvaja vaca west
Tópico #1: él argentino pueblo político país año argentina gobierno política venir
Tópico #2: diputados provincia él razón convenio aconsejar 1984 ricardo ruta alberto
Tópico #3: él decreto constitución diputados juez corte caso congreso justicia electoral
Tópico #4: él impuesto aplicación establecer régimen autoridad establecido servicio caso público
Tópico #5: tema él provincia debate poder tener problema momento presupuesto iniciativa
Tópico #6: derecho él justicia mujer sociedad político derechos país caso libertad
Tópico #7: ciento año gobierno millón argentina él presupuesto deuda pesos pagar
Tópico #8: argentina país educación república argentino universidad internacional año gobierno él
Tópico #9: trabajador trabajo social salud empresa público año laboral

In [1]:
import pandas as pd
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN

# 1. Cargar el dataset inmaculado
df = pd.read_parquet("../data/intervenciones_limpias.parquet")
textos = df['texto_limpio'].dropna().tolist()

print(f"Iniciando BERTopic con {len(textos)} documentos limpios...")

# 2. Configurar modelos subyacentes
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
# UMAP: min_dist=0.0 hace que los clusters sean más densos y claros
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)
# HDBSCAN: min_cluster_size=100 evita que se formen micro-tópicos de 10 frases
hdbscan_model = HDBSCAN(min_cluster_size=100, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    language="spanish",
    calculate_probabilities=False, 
    verbose=True # Para que te imprima en qué paso va (Embeddings -> UMAP -> HDBSCAN)
)

# 3. Entrenar el Monstruo
print("Calculando embeddings y agrupando (esto va a tomar tiempo)...")
topics, probs = topic_model.fit_transform(textos)
 
# 4. GUARDAR EL MODELO (CRUCIAL)
print("¡Terminó! Guardando el modelo para los próximos experimentos...")
topic_model.save("../data/bertopic_diputados_final", serialization="safetensors", save_ctfidf=True)

# 5. Ver los resultados
display(topic_model.get_topic_info().head(20))

Iniciando BERTopic con 172417 documentos limpios...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Calculando embeddings y agrupando (esto va a tomar tiempo)...


2026-06-20 12:22:00,919 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/5389 [00:00<?, ?it/s]

2026-06-20 12:40:23,304 - BERTopic - Embedding - Completed ✓
2026-06-20 12:40:23,305 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-06-20 12:44:38,197 - BERTopic - Dimensionality - Completed ✓
2026-06-20 12:44:38,199 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-20 12:44:50,430 - BERTopic - Cluster - Completed ✓
2026-06-20 12:44:50,448 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-20 12:44:58,289 - BERTopic - Representation - Completed ✓


¡Terminó! Guardando el modelo para los próximos experimentos...


,Topic,Count,Name,Representation,Representative_Docs
0,-1,89116,-1_él_provincia_caso_año,"[él, provincia, caso, año, diputados, derecho,...",[iniciativa síntesis país año chico historia u...
1,0,20261,0_argentino_argentina_país_gobierno,"[argentino, argentina, país, gobierno, pueblo,...",[lugar situar yo planteo sentir necesidad demo...
2,1,2251,1_presupuesto_gasto_ciento_millón,"[presupuesto, gasto, ciento, millón, año, peso...",[resto erogación variar significar ciento gast...
3,2,2224,2_pastoriza_landau_abdala_silvestre,"[pastoriza, landau, abdala, silvestre, soto, o...",[154 agüero diputados 41ª argüello arriaga art...
4,3,1768,3_debate_discusión_tema_debatir,"[debate, discusión, tema, debatir, consenso, s...","[repetir discusión debate año, comenzar debate..."
5,4,1339,4_educación_universidad_docente_educativo,"[educación, universidad, docente, educativo, u...",[intervención dividir lugar referirar discutir...
6,5,1305,5_energía_gas_combustible_combustibles,"[energía, gas, combustible, combustibles, eléc...",[lugar reflexión característica hidrógeno impo...
7,6,1302,6_penal_código_pena_delito,"[penal, código, pena, delito, prisión, procesa...",[franco con­ tradicción norma rango constituci...
8,7,1187,7_afi_127_129_131,"[afi, 127, 129, 131, 132, 130, 133, 128, 147, ...","[129 afi, 129 afi, 127 afi]"
9,8,1102,8_impuesto_ganancia_tributario_fiscal,"[impuesto, ganancia, tributario, fiscal, impos...",[renta suma equivalente ochenta sesenta ciento...
